# Hold a GPU while you work in this notebook

`quickstart.ipynb` books whole calendar days. This one takes a slot for the
length of a working session and gives it back when you are done — the shape you
want when you are sitting at the keyboard rather than queueing a batch job.

Run the cells top to bottom. The last one releases the card, so do not skip it.

In [ ]:
import hrm

hrm.connect(user="cwinkelmann")      # or set HRM_USER / HRM_URL
print("hrm", hrm.__version__)

## What is free right now

`hrm.capacity()` is the one authoritative answer: what a card has free now, and
when the shape you want would next fit. Ask it rather than summing bookings
yourself — the server owns that arithmetic.

In [ ]:
cap = hrm.capacity(memory=8, hours=3)
print("free now: %.1f GiB on GPU %s" % (cap.free_gib, cap.gpu))
print("8 GiB for 3 h would fit from:", cap.earliest)

# An `earliest` in the future does not mean you are refused — it means you
# would be queued, and given that slot.

## Take a slot

An interactive session usually wants **`important=True`**: you are at the
keyboard and cannot checkpoint on command, so nobody should be able to take the
card from under you.

A batch job is the opposite — it books `lendable` with `notice_seconds=0` and
resumes from a checkpoint if it loses the card. That is `examples/train_gpu.py`.

In [ ]:
slot = hrm.reserve(memory=8, hours=3, important=True, note="notebook session")
print(slot)
print("starts later:", slot.assigned_later)

slot.wait_until_start()
print("GPU", slot.gpu, "is yours until", slot.end.astimezone().strftime("%H:%M"))

## Work

The booking holds while you think. Put your own GPU work in the cell below —
this one just allocates a tensor so the booking corresponds to something real.

In [ ]:
try:
    import torch
    x = torch.randn(8192, 8192, device="cuda:%d" % slot.gpu)
    print("%.1f GiB allocated on GPU %d"
          % (torch.cuda.memory_allocated(slot.gpu) / 1024**3, slot.gpu))
except ImportError:
    print("no torch here — this cell is where your own GPU work goes")

## See it on the calendar

Your booking among everyone else's. The dashboard shows the same rows.

In [ ]:
df = hrm.bookings().to_frame()
df[df["id"].isin([b.id for b in slot.bookings])]

## Give it back

**This is the cell people forget.** If you skip it the booking stands until its
end and the card looks taken to everyone else, while sitting idle — the monitor
notices exactly that and says so in `#gpu-watch`, with your name on it.

`release()` moves the booking's end to now rather than deleting it, so the window
you actually used stays on the calendar and in the energy report.

In [ ]:
slot.release()
print("released:", slot.refresh().booking)